In [4]:
import MDAnalysis as md
import pandas as pd
import numpy as np
from scipy.spatial import KDTree
import itertools
from tqdm import tqdm

In [6]:
'''
    Setup the md analysis universe
'''
system_dir = "/home/dp/data/quadromers_180mV_300mM_compressed/_A/ATGC/setup/"
system_psf = system_dir + "system_complete.psf" #For now, we consider the full system
system_pdb = system_dir + "system_complete.pdb"

system_universe = md.Universe(system_psf, system_pdb)
system = system_universe.select_atoms("not water and not name CLA POT")

In [7]:
'''
    Grid related variables
'''
bulk_conductivity = 10.5 #S/m 1M KCl experimental (assuming 1 atm & ~295K)

#These are from the PBC simulation I use (all in A)
size_x = 59
size_y = size_x #x & y should have same size
size_z = 85

#Atom Radii (in A)
radii_dict = dict(zip(pd.read_csv("radial.csv")["Atom_type"], pd.read_csv("radial.csv")["radius(A)"]))

#Radial linear conductance variables (from https://pubs.acs.org/doi/10.1021/acssensors.8b01375)
r_min = 1.4 #A
r_slope_inv = 2.9 #A (this should be divided for actual slope)

#Useful intermediate vars
r_intercept = -1.0 * r_min / r_slope_inv
r_max = r_slope_inv + r_min

#For conductivity map, just use np.clip(dist / r_slope_inv + r_intercept, 0, 1)

In [8]:
#Write the radii for each atom and add as attribute to 
names = np.strings.slice(system.names.astype(np.dtypes.StringDType), 0,1)
radii = np.array([radii_dict[k] for k in names])

In [11]:
'''
    Construct the grid and add the conductances based on the closest atom
'''

#Grid spacing along dimensions
x_arr = np.arange(-1.0 * int(size_x / 2), 1.0 * int(size_x / 2), 1, dtype=int)
y_arr = np.arange(-1.0 * int(size_y / 2), 1.0 * int(size_y / 2), 1, dtype=int)
z_arr = np.arange(-1.0 * int(size_z / 2), 1.0 * int(size_z / 2), 1, dtype=int)

#Will contain the conductances
grid = np.empty((len(x_arr), len(y_arr), len(z_arr)))

#Method for having nice tqdm progress bar
coords = list(itertools.product(x_arr,y_arr,z_arr))
pg = tqdm(coords)

#Generate the KDTree
kdt = KDTree(system.positions)

#Iterate through grid points
for x,y,z in pg:

    #Setup position
    pos = (x,y,z)

    #Do KDTree query
    query = kdt.query_ball_point(pos, r=(r_max + 1.9)) #1.9 comes from the radial.csv (will be the maximum distance in which conductance isn't just bulk)
    query_pos = system.positions[query]
    
    #If there are no neighbors within the maximum distance range, then set to bulk
    if(len(query_pos) == 0):
        grid[x,y,z] = bulk_conductivity
    else:
        #Compute the distance for all possible points and readjust using the atom's radius
        query_dist = np.linalg.norm(query_pos - pos)
        real_dist = query_dist - radii[query]
        selected_dist = real_dist[np.argmax(real_dist)]

        #Set the conductivity according to the linear piecewise as in the paper
        grid[x,y,z] = bulk_conductivity * np.clip(selected_dist / r_slope_inv + r_intercept, 0, 1)


100%|██████████████████████████████████████████████████████████████████████████████████| 282576/282576 [00:35<00:00, 8002.13it/s]


In [177]:
'''
    Construct the grid and add the conductances based on the closest atom (this is using the tree query method)
'''

#Grid spacing along dimensions
x_arr = np.arange(-1.0 * int(size_x / 2), 1.0 * int(size_x / 2), 1, dtype=int)
y_arr = np.arange(-1.0 * int(size_y / 2), 1.0 * int(size_y / 2), 1, dtype=int)
z_arr = np.arange(-1.0 * int(size_z / 2), 1.0 * int(size_z / 2), 1, dtype=int)

#Will contain the conductances
grid = np.empty((len(x_arr), len(y_arr), len(z_arr)))

#Method for having nice tqdm progress bar
coords = np.array(list(itertools.product(x_arr,y_arr,z_arr)))

#Generate the KDTree
tree_pos = KDTree(system.positions)
tree_grid = KDTree(coords)

#Make the query, returns list of nearest neighbor for every point 
query = tree_grid.query_ball_tree(tree_pos, r=(r_max + 1.9)) #This does all querys between the two trees

#Reformat to a numpy via a pandas dataframe for ease of use since the list is inhomogenous originally
query_np = pd.DataFrame(query).to_numpy()

#Will store final conductivity map
conductivity_map = np.empty((len(query), 1))

#For any point that has nothing within maximum distance, we set to max bulk
conductivity_map[np.all(np.isnan(query_np), axis=1)] = bulk_conductivity

In [108]:
#Now we deal with the ones with a maximum distance
indices=np.arange(0,query_np.shape[0]) #Used to get the indices 
used_indices = []

for i in tqdm(reversed(range(query_np.shape[1]))):
    mask = np.all(~np.isnan(query_np[:,:i]), axis=1)
    print(len(query_np[mask]))

15it [00:00, 73.85it/s]

1
1
1
1
2
8
12
17
23
31
44
70
92
156
235
361


32it [00:00, 78.10it/s]

551
798
1103
1511
2028
2674
3517
4428
5492
6757
8164
9782
11402
13216
15162
17130
19207


40it [00:00, 72.17it/s]

...

165654
167495
169417
171442
173476
175603


131it [00:03, 29.36it/s]

177886
180262
183497
186186
189548
195116


134it [00:03, 41.03it/s]

282576


In [178]:
#This is how the update should go, then we remove the used querys and reduce the length by one

mask = np.all(~np.isnan(query_np), axis=1)

#Compute the distances from given grid coords
dist = np.linalg.norm(system.positions[query_np[mask].astype(int)] - coords[mask], axis=2)
real_dist = dist - radii[query_np[mask].astype(int)] #Correct the distances with the atom's surface

#Grab the smallest distance indices
shortest_index = np.argmin(real_dist, axis=1)

#Update the respective conductivities
conductivity_map[mask] = np.clip(real_dist[:,shortest_index] / r_slope_inv + r_intercept, 0, 1)